In [1]:
%matplotlib inline
import os
import sys
sys.path.append('/Users/shampa/software/anaconda3/envs/py312/lib/python3.12/site-packages')
import matplotlib.pyplot as plt

from ipywidgets import interact,fixed,IntSlider
import ipywidgets

from io import BytesIO
import pandas as pd
import numpy as np
from rdkit.Chem import PandasTools

from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem import DataStructs
from rdkit.Chem import rdMolDescriptors
from rdkit.Chem import rdRGroupDecomposition
from rdkit.Chem import rdMolTransforms

from rdkit.Chem.Draw import IPythonConsole #Needed to show molecules
from rdkit.Chem import Draw
from rdkit.Chem import rdDepictor
from rdkit.Chem.Draw import rdMolDraw2D
from rdkit.Chem.Draw.MolDrawing import MolDrawing, DrawingOptions #Only needed if modifying defaults

DrawingOptions.bondLineWidth=1.8
IPythonConsole.ipython_useSVG=True
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.warning')
import rdkit
import py3Dmol
print(rdkit.__version__)

2024.03.3


In [2]:
def MolTo3DView(mol, size=(300, 300), style="stick", surface=False, opacity=0.5):
    """Draw molecule in 3D
    
    Args:
    ----
        mol: rdMol, molecule to show
        size: tuple(int, int), canvas size
        style: str, type of drawing molecule
               style can be 'line', 'stick', 'sphere', 'carton'
        surface, bool, display SAS
        opacity, float, opacity of surface, range 0.0-1.0
    Return:
    ----
        viewer: py3Dmol.view, a class for constructing embedded 3Dmol.js views in ipython notebooks.
    """
    assert style in ('line', 'stick', 'sphere', 'carton')
    mblock = Chem.MolToMolBlock(mol)
    viewer = py3Dmol.view(width=size[0], height=size[1])
    viewer.addModel(mblock, 'mol')
    viewer.setStyle({style:{}})
    if surface:
        viewer.addSurface(py3Dmol.SAS, {'opacity': opacity})
    viewer.zoomTo()
    return viewer

In [3]:
def smi2conf_etkdg(smiles):
    '''Convert SMILES to rdkit.Mol with 3D coordinates
       Use Experimental Torsion angles Knowledge-based Distance Geometry ETKDGv3
       method - version 3 (small rings)
       params.numThreads = 0 uses max no. of threads allowed on a conputer
    '''
    params = Chem.rdDistGeom.srETKDGv3()
    params.randomSeed = 12412
    params.clearConfs = True
    params.numThreads = 0
    
    mol = Chem.MolFromSmiles(smiles)
    if mol is not None:
        mol = Chem.AddHs(mol)
        AllChem.EmbedMolecule(mol)
        AllChem.EmbedMultipleConfs(mol, numConfs=10, params=params)
        
        return mol
    else:
        return None

In [5]:
def enumerateTorsions(mol):
    torsionList = []
    
    # Iterate through all bonds in the molecule
    for bond in mol.GetBonds():
        beginAtomIdx = bond.GetBeginAtomIdx()
        endAtomIdx = bond.GetEndAtomIdx()
        
        # Get the atoms connected by the current bond
        beginAtom = mol.GetAtomWithIdx(beginAtomIdx)
        endAtom = mol.GetAtomWithIdx(endAtomIdx)
        
        # Skip bonds involving terminal atoms with valency 1
        if beginAtom.GetExplicitValence() == 1 or endAtom.GetExplicitValence() == 1 or beginAtom.GetExplicitValence() == 2 or endAtom.GetExplicitValence() == 2 or beginAtom.GetExplicitValence() == 3 or endAtom.GetExplicitValence() == 3:
            continue
        
        # Get neighboring atoms for beginAtom and endAtom to form torsions
        for neighbor1 in beginAtom.GetNeighbors():
            if neighbor1.GetIdx() != endAtom.GetIdx():
                atom1 = neighbor1
                break
        for neighbor2 in endAtom.GetNeighbors():
            if neighbor2.GetIdx() != beginAtom.GetIdx():
                atom4 = neighbor2
                break
        
        # Append torsion angle indices (atom indices) to the torsionList
        torsionList.append((atom1.GetIdx(), beginAtomIdx, endAtomIdx, atom4.GetIdx()))
    
    return torsionList


## TESTing for a single SMILES

In [6]:
smiles = 'CC'
mol = Chem.MolFromSmiles(smiles)
mol_h = Chem.AddHs(mol)
params = Chem.rdDistGeom.srETKDGv3()
params.randomSeed = 12412
params.clearConfs = True

torsionList = enumerateTorsions(mol_h)
print(f"List of torsions: {torsionList}")


cids = AllChem.EmbedMultipleConfs(mol_h, numConfs=100, params=params)

atoms=[a for a in mol_h.GetAtoms()]
for a in atoms:
    print( a.GetIdx(), a.GetSymbol() )


bonds = [(x.GetBeginAtomIdx(), x.GetEndAtomIdx()) for x in mol_h.GetBonds()]
print(bonds)


for cidx in cids:
    print(f"Conformer ID: {cidx}")
    conf = mol_h.GetConformer(cidx)
    torsionList = enumerateTorsions(mol_h)
    
    # Print dihedral angle for each torsion in degrees
    for torsion in torsionList:
        i, j, k, l = torsion
        dihedral_angle = rdMolTransforms.GetDihedralDeg(conf, i, j, k, l)
        print(f"Dihedral angle for torsion {torsion}: {dihedral_angle} degrees")

List of torsions: [(2, 0, 1, 5)]
0 C
1 C
2 H
3 H
4 H
5 H
6 H
7 H
[(0, 1), (0, 2), (0, 3), (0, 4), (1, 5), (1, 6), (1, 7)]
Conformer ID: 0
Dihedral angle for torsion (2, 0, 1, 5): -88.61292004104415 degrees
Conformer ID: 1
Dihedral angle for torsion (2, 0, 1, 5): 59.02857668439743 degrees
Conformer ID: 2
Dihedral angle for torsion (2, 0, 1, 5): 99.29521651388137 degrees
Conformer ID: 3
Dihedral angle for torsion (2, 0, 1, 5): 105.04866889787807 degrees
Conformer ID: 4
Dihedral angle for torsion (2, 0, 1, 5): -57.31607958765178 degrees
Conformer ID: 5
Dihedral angle for torsion (2, 0, 1, 5): 131.47344719910762 degrees
Conformer ID: 6
Dihedral angle for torsion (2, 0, 1, 5): 89.85999761326282 degrees
Conformer ID: 7
Dihedral angle for torsion (2, 0, 1, 5): 82.11880625340811 degrees
Conformer ID: 8
Dihedral angle for torsion (2, 0, 1, 5): 145.75946993472806 degrees
Conformer ID: 9
Dihedral angle for torsion (2, 0, 1, 5): 52.827199534989575 degrees
Conformer ID: 10
Dihedral angle for torsio

## Loading train dataset

In [13]:
df_train=pd.read_csv('./qm9.csv')
df_train_smiles=df_train["smiles"]
df_train_smiles.head(5)
print(len(df_train_smiles))

1000


In [14]:
confs = [smi2conf_etkdg(s) for s in df_train_smiles]
max_confs=len(confs)

def style_selector(idx, s):
    conf = confs[idx]
    return MolTo3DView(conf, style=s).show()

interact(style_selector, 
         idx=ipywidgets.IntSlider(min=0,max=(max_confs-1), step=1),
         s=ipywidgets.Dropdown(
            options=['line', 'stick', 'sphere'],
            value='line',
            description='Style:'))

interactive(children=(IntSlider(value=0, description='idx', max=999), Dropdown(description='Style:', options=(…

<function __main__.style_selector(idx, s)>

## Loading test dataset

In [15]:
df_test=pd.read_csv('./test.csv')
df_test_smiles=df_test["smiles"]
df_test_smiles.head(5)
print([df_test_smiles])
print(len(df_test_smiles))

[0             CC1=C(F)N=CNC1=N
1              COC1CCC2(CO2)C1
2             C#CC12OC1C1CCC21
3            CC1=CC(O)C(C1)=NO
4                 COC(C)C(C)CO
                ...           
95            C1CC2(COC2)OC1=O
96    COC1CC1([NH3+])C([O-])=O
97           CC(C)C1(C)OC1(C)C
98             O=C1C2CC=CCC1O2
99              CC(=O)C(CO)C#N
Name: smiles, Length: 100, dtype: object]
100


In [11]:
confs = [smi2conf_etkdg(s) for s in df_test_smiles]
max_confs=len(confs)

def style_selector(idx, s):
    conf = confs[idx]
    return MolTo3DView(conf, style=s).show()

interact(style_selector, 
         idx=ipywidgets.IntSlider(min=0,max=(max_confs-1), step=1),
         s=ipywidgets.Dropdown(
            options=['line', 'stick', 'sphere'],
            value='line',
            description='Style:'))

interactive(children=(IntSlider(value=0, description='idx', max=742), Dropdown(description='Style:', options=(…

<function __main__.style_selector(idx, s)>

## Checking a specific SMILES Using MMFF94s

In [ ]:
def smi2conf(smiles):
    '''Convert SMILES to rdkit.Mol with 3D coordinates'''
    mol = Chem.MolFromSmiles(smiles)
    if mol is not None:
        mol = Chem.AddHs(mol)
        AllChem.EmbedMolecule(mol)
        AllChem.MMFFOptimizeMolecule(mol, maxIters=200, mmffVariant='MMFF94s')
        return mol
    else:
        return None

@interact
def smi2viewer(smi='OO'):
    try:
        conf = smi2conf(smi)
        return MolTo3DView(conf).show()
    except:
        return None